# Aula Prática — Particionamento Customizado e Processamento de Streams
### Computação Paralela e Distribuída (DCE540)

Esta aula conecta dois assuntos que, à primeira vista, parecem distintos, mas se encaixam naturalmente:

1. **Particionamento customizado** — controlar exatamente *para onde* cada mensagem vai, em vez de deixar isso só por conta do hash da chave;
2. **Processamento de streams** (os conceitos por trás do **Kafka Streams**) — o que fazer *continuamente* com os dados depois que eles já estão bem organizados nas partições certas.

**Nota sobre a biblioteca usada:** o Kafka Streams "oficial" é uma biblioteca **Java** — não existe um cliente Python equivalente mantido pela própria Apache. Para manter a consistência com as aulas anteriores, vamos **implementar os conceitos centrais manualmente em Python** (KTable, agregação contínua, janelas de tempo, joins), usando só o `kafka-python` que já conhecemos. A Seção 7, ao final, mostra como isso ficaria com o Kafka Streams de verdade (em Java), e menciona uma alternativa Python que roda de fato.

**Pré-requisito:** aulas anteriores de Kafka (tópicos, partições, consumer group, semânticas de entrega).


## 0. Preparando o ambiente (VS Code + Docker)

Ambiente novo e independente das aulas anteriores. Siga estes passos:

1. Confirme que o **VS Code** tem as extensões **Python** e **Jupyter** instaladas.
2. Confirme que o **Docker Desktop** está aberto e rodando.
3. Se algum ambiente de aula anterior ainda estiver rodando, pare-o primeiro (mesmas portas 9092, 9094, 8080):

   ```bash
   # na pasta da aula anterior que ainda estiver de pé
   docker compose down
   ```

4. Abra a pasta **desta aula** no VS Code — `docker-compose.yml`, este notebook e `requirements.txt` devem estar juntos.
5. Em um terminal do VS Code, crie o ambiente virtual e instale as dependências:

   **Windows (PowerShell):**
   ```powershell
   python -m venv venv
   venv\Scripts\activate
   python -m pip install --upgrade pip
   python -m pip install -r requirements.txt
   ```

   **Linux/Mac:**
   ```bash
   python3 -m venv venv
   source venv/bin/activate
   python3 -m pip install --upgrade pip
   python3 -m pip install -r requirements.txt
   ```

   > **Use sempre `python -m pip install ...`, nunca `pip install ...` sozinho** — evita que os pacotes sejam instalados no Python global em vez do venv.

6. Abra o notebook no VS Code e selecione o kernel `venv`.
7. No terminal, suba o ambiente:

   ```bash
   docker compose up -d
   ```

   Confira com `docker compose ps` — `kafka-aula4` e `kafka-ui-aula4` devem aparecer com status `running`.

8. (Opcional) Acompanhe em `http://localhost:8080` (Kafka UI) — útil especialmente na Seção 2, para visualizar a distribuição de mensagens entre partições.

Ao final da aula:

```bash
docker compose down -v
```


## 1. Contextualização — por que particionar de forma customizada?

Por padrão, o Kafka decide a partição de duas formas: sem chave (`key=None`), distribui round-robin/sticky; com chave, aplica hash da chave (`murmur2`) módulo o número de partições.

Isso não é suficiente em alguns cenários:

- **Carga desbalanceada por chave** — se uma fração pequena de chaves concentra a maior parte do tráfego, o hash padrão empilha tudo numa única partição ("hot partition"), limitando o paralelismo mesmo com várias partições disponíveis;
- **Roteamento/prioridade** — você pode querer que um tipo específico de mensagem (ex.: pedidos urgentes) sempre caia num subconjunto dedicado de partições, para que consumidores possam priorizá-las;
- **Localidade** — otimizar afinidade de cache num consumidor downstream, agrupando mensagens relacionadas de forma diferente do que o hash padrão faria.

**Regra de ouro ao implementar isso:** a lógica precisa continuar **determinística** (mesma entrada → mesma partição), senão você perde a garantia de ordenação de eventos relacionados dentro de uma partição — uma das poucas garantias de ordem que o Kafka realmente oferece.


## 2. Implementando um particionador customizado

Vamos implementar um cenário concreto: um tópico de pedidos onde os classificados como `"urgente"` sempre vão para uma partição dedicada (partição 0), enquanto os demais são distribuídos pelas partições restantes, por hash determinístico do `cliente_id`.

O `kafka-python` permite isso via uma subclasse de `kafka.partitioner.Partitioner`, passada ao `KafkaProducer` através do parâmetro `partitioner`.


In [1]:
from kafka import KafkaProducer, KafkaConsumer, KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError
from kafka.partitioner import Partitioner
import json
import hashlib
import random
import time
from collections import Counter

BOOTSTRAP_SERVERS = "localhost:9094"
TOPIC = "pedidos-prioridade"
NUM_PARTICOES = 4

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP_SERVERS)
try:
    admin.create_topics([NewTopic(name=TOPIC, num_partitions=NUM_PARTICOES, replication_factor=1)])
    print(f"Tópico '{TOPIC}' criado com {NUM_PARTICOES} partições.")
except TopicAlreadyExistsError:
    print(f"Tópico '{TOPIC}' já existia.")
admin.close()


Tópico 'pedidos-prioridade' criado com 4 partições.


In [2]:
class PartitionadorPorPrioridade(Partitioner):
    """Reserva a partição 0 exclusivamente para pedidos urgentes.
    Os demais pedidos são distribuídos pelas partições restantes, usando um
    hash determinístico da chave (cliente_id) — mesma chave sempre cai na
    mesma partição, DESDE QUE a prioridade também não mude.
    """
    PARTICAO_URGENTE = 0

    def partition(self, topic, key, serialized_key, value, serialized_value, cluster):
        todas_particoes = sorted(cluster.partitions_for_topic(topic))
        if len(todas_particoes) <= 1:
            return todas_particoes[0]

        # 'value' aqui é o dict ORIGINAL (antes da serialização) passado a producer.send()
        eh_urgente = isinstance(value, dict) and value.get("prioridade") == "urgente"
        if eh_urgente:
            return self.PARTICAO_URGENTE

        particoes_normais = [p for p in todas_particoes if p != self.PARTICAO_URGENTE]
        if serialized_key is None:
            return random.choice(particoes_normais)

        indice = int(hashlib.md5(serialized_key).hexdigest(), 16) % len(particoes_normais)
        return particoes_normais[indice]


print("Particionador customizado definido.")


Particionador customizado definido.


**Nota importante:** repare que a mesma chave (`cliente_id`) pode cair em partições diferentes dependendo da prioridade do pedido — isso é uma escolha deliberada deste particionador (priorizar roteamento por urgência acima da ordenação por cliente). Vale sempre deixar esse tipo de trade-off explícito, porque quebra parcialmente a expectativa usual de "mesma chave, mesma partição".

### Publicando uma carga mista (normal + urgente) com o particionador customizado


In [3]:
def publicar_pedidos_prioridade(producer, n):
    clientes = [f"cliente-{i}" for i in range(5)]
    for i in range(n):
        pedido = {
            "cliente_id": random.choice(clientes),
            "valor": round(random.uniform(10, 500), 2),
            "prioridade": "urgente" if random.random() < 0.2 else "normal",  # ~20% urgentes
        }
        producer.send(TOPIC, key=pedido["cliente_id"].encode(), value=pedido)
    producer.flush()


producer_customizado = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    key_serializer=lambda k: k,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    partitioner=PartitionadorPorPrioridade(),
)

publicar_pedidos_prioridade(producer_customizado, 40)
producer_customizado.close()
print("40 pedidos publicados com o particionador customizado.")


40 pedidos publicados com o particionador customizado.


### Conferindo a distribuição resultante


In [4]:
def contar_mensagens_por_particao(group_id):
    consumer = KafkaConsumer(
        TOPIC,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=group_id,
        auto_offset_reset="earliest",
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms=5000,
    )
    contagem_particao = Counter()
    contagem_urgentes_por_particao = Counter()
    for msg in consumer:
        contagem_particao[msg.partition] += 1
        if msg.value.get("prioridade") == "urgente":
            contagem_urgentes_por_particao[msg.partition] += 1
    consumer.close()
    return contagem_particao, contagem_urgentes_por_particao

distribuicao, urgentes_por_particao = contar_mensagens_por_particao("grupo-analise-customizado")
print("Mensagens por partição:", dict(sorted(distribuicao.items())))
print("Urgentes por partição:  ", dict(sorted(urgentes_por_particao.items())))


/tmp/ipykernel_153/2753189921.py:2: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Mensagens por partição: {0: 7, 1: 7, 2: 18, 3: 8}
Urgentes por partição:   {0: 7}


Se o particionador funcionou como esperado, **toda** mensagem urgente deve estar na partição 0, e nenhuma mensagem urgente deve aparecer nas partições 1, 2 ou 3.

## 3. Comparação com o particionador padrão

Vamos publicar uma carga equivalente, mas usando o particionador **padrão** do Kafka (hash de chave, sem distinção de prioridade), num tópico separado, para comparar a distribuição.


In [5]:
TOPIC_PADRAO = "pedidos-prioridade-padrao"

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP_SERVERS)
try:
    admin.create_topics([NewTopic(name=TOPIC_PADRAO, num_partitions=NUM_PARTICOES, replication_factor=1)])
    print(f"Tópico '{TOPIC_PADRAO}' criado.")
except TopicAlreadyExistsError:
    print(f"Tópico '{TOPIC_PADRAO}' já existia.")
admin.close()

producer_padrao = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    key_serializer=lambda k: k,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    # sem 'partitioner' — usa o DefaultPartitioner (hash de chave)
)

random.seed(42)  # mesma "carga" da Seção 2, para comparação justa
clientes = [f"cliente-{i}" for i in range(5)]
for i in range(40):
    pedido = {
        "cliente_id": random.choice(clientes),
        "valor": round(random.uniform(10, 500), 2),
        "prioridade": "urgente" if random.random() < 0.2 else "normal",
    }
    producer_padrao.send(TOPIC_PADRAO, key=pedido["cliente_id"].encode(), value=pedido)
producer_padrao.flush()
producer_padrao.close()

consumer = KafkaConsumer(
    TOPIC_PADRAO,
    bootstrap_servers=BOOTSTRAP_SERVERS,
    group_id="grupo-analise-padrao",
    auto_offset_reset="earliest",
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    consumer_timeout_ms=5000,
)
urgentes_padrao = Counter()
for msg in consumer:
    if msg.value.get("prioridade") == "urgente":
        urgentes_padrao[msg.partition] += 1
consumer.close()

print("Urgentes por partição (particionador PADRÃO):", dict(sorted(urgentes_padrao.items())))
print("Compare com 'Urgentes por partição' da Seção 2 — no padrão, ficam espalhadas; no customizado, concentradas na partição 0.")


Tópico 'pedidos-prioridade-padrao' criado.


/tmp/ipykernel_153/1380929174.py:30: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Urgentes por partição (particionador PADRÃO): {0: 4, 2: 2, 3: 7}
Compare com 'Urgentes por partição' da Seção 2 — no padrão, ficam espalhadas; no customizado, concentradas na partição 0.


## 4. De particionamento a processamento de stream

Com os dados bem organizados, a próxima pergunta natural é: **o que fazer continuamente com esse fluxo?** É essa a proposta do **Kafka Streams** — processar dados que estão fluindo por tópicos, transformando entrada em saída, continuamente, sem esperar um "job em lote" periódico.

Três conceitos centrais (já vistos em texto, agora implementados na prática):

- **KTable** — uma tabela sempre atualizada a partir de um tópico: cada nova mensagem com uma chave **substitui** o valor anterior daquela chave (representa "o estado atual de X");
- **Agregação contínua** — um valor (soma, contagem, média) atualizado incrementalmente a cada mensagem nova, sem esperar o fim de um período;
- **Janelas de tempo (windowing)** — agregações restritas a um intervalo (ex.: "só os últimos 10 segundos"), necessárias quando "o total desde sempre" não é o que se quer.

Vamos implementar cada um manualmente, em Python puro, sobre o que o `kafka-python` já nos dá.

### 4.1 KTable — última atualização por chave


In [6]:
TOPIC_CADASTRO = "clientes-cadastro"

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP_SERVERS)
try:
    admin.create_topics([NewTopic(name=TOPIC_CADASTRO, num_partitions=1, replication_factor=1)])
    print(f"Tópico '{TOPIC_CADASTRO}' criado.")
except TopicAlreadyExistsError:
    print(f"Tópico '{TOPIC_CADASTRO}' já existia.")
admin.close()

# Publicando um "cadastro" de clientes — repare que cliente-1 aparece duas vezes,
# a segunda mensagem deve SUBSTITUIR a primeira (comportamento de KTable).
cadastros = [
    {"cliente_id": "cliente-0", "nome": "Ana",   "cidade": "Alfenas"},
    {"cliente_id": "cliente-1", "nome": "Bruno",  "cidade": "Poços de Caldas"},
    {"cliente_id": "cliente-2", "nome": "Carla",  "cidade": "Varginha"},
    {"cliente_id": "cliente-3", "nome": "Diego",  "cidade": "Pouso Alegre"},
    {"cliente_id": "cliente-4", "nome": "Eva",    "cidade": "Machado"},
    {"cliente_id": "cliente-1", "nome": "Bruno Costa", "cidade": "Belo Horizonte"},  # update!
]

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    key_serializer=lambda k: k.encode("utf-8"),
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)
for c in cadastros:
    producer.send(TOPIC_CADASTRO, key=c["cliente_id"], value=c)
producer.flush()
producer.close()


def consumir_como_ktable(topico, group_id):
    """Lê um tópico do início ao fim e devolve um dict {chave: último_valor} —
    é exatamente essa reinterpretação de um tópico como 'changelog' que o
    Kafka Streams chama de KTable."""
    consumer = KafkaConsumer(
        topico,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=group_id,
        auto_offset_reset="earliest",
        key_deserializer=lambda k: k.decode("utf-8"),
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms=5000,
    )
    tabela = {}
    for msg in consumer:
        tabela[msg.key] = msg.value   # a mensagem mais recente SEMPRE sobrescreve
    consumer.close()
    return tabela

cadastro_ktable = consumir_como_ktable(TOPIC_CADASTRO, "grupo-ktable-clientes")
for cliente_id, dados in cadastro_ktable.items():
    print(cliente_id, "->", dados)


Tópico 'clientes-cadastro' criado.


/tmp/ipykernel_153/567288829.py:37: DeprecationWarning: key_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(
/tmp/ipykernel_153/567288829.py:37: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


cliente-0 -> {'cliente_id': 'cliente-0', 'nome': 'Ana', 'cidade': 'Alfenas'}
cliente-1 -> {'cliente_id': 'cliente-1', 'nome': 'Bruno Costa', 'cidade': 'Belo Horizonte'}
cliente-2 -> {'cliente_id': 'cliente-2', 'nome': 'Carla', 'cidade': 'Varginha'}
cliente-3 -> {'cliente_id': 'cliente-3', 'nome': 'Diego', 'cidade': 'Pouso Alegre'}
cliente-4 -> {'cliente_id': 'cliente-4', 'nome': 'Eva', 'cidade': 'Machado'}


Repare que `cliente-1` aparece só uma vez no resultado final, com os dados **mais recentes** (`"Bruno Costa"`, `"Belo Horizonte"`) — exatamente o comportamento de uma KTable.

### 4.2 Agregação contínua (stateful)

Isso já apareceu, sem esse nome, nas aulas anteriores (total por sensor). Aqui só formalizamos: cada mensagem nova **atualiza incrementalmente** um estado acumulado, em vez de recalcular tudo do zero.


In [7]:
def agregar_valor_total_por_cliente(topico, group_id):
    consumer = KafkaConsumer(
        topico,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=group_id,
        auto_offset_reset="earliest",
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms=5000,
    )
    total_por_cliente = {}   # o "state store" da agregação, em memória
    for msg in consumer:
        pedido = msg.value
        cliente = pedido["cliente_id"]
        total_por_cliente[cliente] = total_por_cliente.get(cliente, 0.0) + pedido["valor"]
    consumer.close()
    return total_por_cliente

totais = agregar_valor_total_por_cliente(TOPIC, "grupo-agregacao-continua")
for cliente, total in sorted(totais.items()):
    print(f"{cliente}: R$ {total:.2f}")


/tmp/ipykernel_153/2243499802.py:2: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


cliente-0: R$ 4246.90
cliente-1: R$ 3429.92
cliente-2: R$ 1189.23
cliente-3: R$ 1537.76
cliente-4: R$ 1046.66


### 4.3 Janela de tempo (tumbling window)

Agora, em vez de "o total desde sempre", queremos "o total de cada intervalo fixo de tempo" — uma **tumbling window** (janelas fixas, sem sobreposição). Como nosso tópico não tem um timestamp de evento embutido nos dados, vamos usar o próprio `timestamp` da mensagem no Kafka (`msg.timestamp`, em milissegundos).


In [8]:
TAMANHO_JANELA_MS = 2000  # janelas de 2 segundos, só para tornar o exemplo visível rapidamente

def agregar_por_janela(topico, group_id, tamanho_janela_ms):
    consumer = KafkaConsumer(
        topico,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=group_id,
        auto_offset_reset="earliest",
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms=5000,
    )
    totais_por_janela = {}   # {inicio_da_janela_ms: total}
    for msg in consumer:
        pedido = msg.value
        inicio_janela = (msg.timestamp // tamanho_janela_ms) * tamanho_janela_ms
        totais_por_janela[inicio_janela] = totais_por_janela.get(inicio_janela, 0.0) + pedido["valor"]
    consumer.close()
    return dict(sorted(totais_por_janela.items()))

# Publicamos os 40 pedidos da Seção 2 de uma vez (bem rápido), então é esperado
# que a maioria caia numa única janela de 2s — o interessante aqui é o MECANISMO
# de bucketing, que você vai reaproveitar com dados mais espaçados na Atividade 2.
totais_janela = agregar_por_janela(TOPIC, "grupo-janela-tumbling", TAMANHO_JANELA_MS)
for inicio, total in totais_janela.items():
    print(f"Janela iniciando em {inicio}: R$ {total:.2f}")


/tmp/ipykernel_153/624475942.py:4: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Janela iniciando em 1790215656000: R$ 11450.47


### 4.4 Join KStream-KTable — enriquecendo um evento com o estado atual de uma tabela

Um padrão muito comum: um evento chega "cru" (só com um ID), e você quer **enriquecê-lo** com dados de referência mantidos em outro lugar — exatamente o que um `KStream.join(KTable)` faz no Kafka Streams, sem precisar de nenhuma janela de tempo (porque a KTable representa "o estado agora", não uma sequência de eventos).


In [9]:
def enriquecer_pedidos_com_cadastro(topico_pedidos, group_id, ktable_clientes):
    consumer = KafkaConsumer(
        topico_pedidos,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=group_id,
        auto_offset_reset="earliest",
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms=5000,
    )
    pedidos_enriquecidos = []
    for msg in consumer:
        pedido = msg.value
        cliente_info = ktable_clientes.get(pedido["cliente_id"])
        pedido_enriquecido = {
            **pedido,
            "nome_cliente": cliente_info["nome"] if cliente_info else None,
            "cidade_cliente": cliente_info["cidade"] if cliente_info else None,
        }
        pedidos_enriquecidos.append(pedido_enriquecido)
    consumer.close()
    return pedidos_enriquecidos

pedidos_enriquecidos = enriquecer_pedidos_com_cadastro(TOPIC, "grupo-join-enriquecimento", cadastro_ktable)
for p in pedidos_enriquecidos[:5]:
    print(p)
print(f"... ({len(pedidos_enriquecidos)} pedidos enriquecidos no total)")


/tmp/ipykernel_153/2903756962.py:2: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


{'cliente_id': 'cliente-1', 'valor': 295.93, 'prioridade': 'normal', 'nome_cliente': 'Bruno Costa', 'cidade_cliente': 'Belo Horizonte'}
{'cliente_id': 'cliente-1', 'valor': 272.77, 'prioridade': 'normal', 'nome_cliente': 'Bruno Costa', 'cidade_cliente': 'Belo Horizonte'}
{'cliente_id': 'cliente-1', 'valor': 259.01, 'prioridade': 'normal', 'nome_cliente': 'Bruno Costa', 'cidade_cliente': 'Belo Horizonte'}
{'cliente_id': 'cliente-1', 'valor': 467.74, 'prioridade': 'normal', 'nome_cliente': 'Bruno Costa', 'cidade_cliente': 'Belo Horizonte'}
{'cliente_id': 'cliente-1', 'valor': 386.16, 'prioridade': 'normal', 'nome_cliente': 'Bruno Costa', 'cidade_cliente': 'Belo Horizonte'}
... (40 pedidos enriquecidos no total)


Repare que o "join" aqui é só uma consulta a um dicionário Python — mas conceitualmente é exatamente o que o Kafka Streams faz por baixo dos panos: manter a KTable materializada localmente (num state store, geralmente RocksDB) e consultá-la a cada evento do KStream, sem nenhuma chamada de rede adicional por mensagem.


## 5. Atividades Práticas

As três atividades a seguir combinam particionamento customizado e processamento de stream. Escreva sua solução na célula de código logo abaixo de cada enunciado.


### Atividade 1 — Particionador por faixa de valor

**Objetivo:** implementar uma segunda estratégia de particionamento customizado, agora baseada em uma faixa de valores em vez de uma flag booleana de prioridade.

**Tarefa:**
1. Crie um novo tópico `pedidos-vip` com 4 partições.
2. Implemente uma classe `PartitionadorPorValor(Partitioner)` que:
   - Reserva a partição 0 para pedidos com `valor > 1000` (chame-os de "VIP");
   - Distribui os demais pedidos (`valor <= 1000`) pelas partições restantes, por hash da chave (`cliente_id`) — igual fizemos na Seção 2.
3. Publique 40 pedidos (reaproveite a lógica de `publicar_pedidos_prioridade`, adaptando o campo de decisão de `prioridade` para `valor`) usando esse novo particionador.
4. Confira a distribuição resultante (reaproveite `contar_mensagens_por_particao`, adaptando a condição de filtro de `"prioridade" == "urgente"` para `valor > 1000`).

**Pergunta para o relatório:** esse particionador por faixa de valor tem alguma desvantagem que o particionador por prioridade (Seção 2) não tinha? (Dica: pense no que acontece se, um dia, a maioria dos pedidos passar a ter valor acima de 1000.)


In [10]:
# Atividade 1
TOPIC_VIP = "pedidos-vip"


class PartitionadorPorValor(Partitioner):
    PARTICAO_VIP = 0

    def partition(self, topic, key, serialized_key, value, serialized_value, cluster):
        particoes = sorted(cluster.partitions_for_topic(topic))

        if len(particoes) == 1:
            return particoes[0]

        if isinstance(value, dict) and value.get("valor", 0) > 1000:
            return self.PARTICAO_VIP

        particoes_normais = [p for p in particoes if p != self.PARTICAO_VIP]
        if serialized_key is None:
            return random.choice(particoes_normais)

        indice = int(hashlib.md5(serialized_key).hexdigest(), 16) % len(particoes_normais)
        return particoes_normais[indice]


admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP_SERVERS)
try:
    admin.create_topics([
        NewTopic(name=TOPIC_VIP, num_partitions=NUM_PARTICOES, replication_factor=1)
    ])
    print(f"Tópico '{TOPIC_VIP}' criado.")
except TopicAlreadyExistsError:
    print(f"Tópico '{TOPIC_VIP}' já existia.")
admin.close()

producer_vip = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    key_serializer=lambda k: k,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    partitioner=PartitionadorPorValor(),
)

random.seed(8)
clientes = [f"cliente-{i}" for i in range(5)]
for _ in range(40):
    pedido = {
        "cliente_id": random.choice(clientes),
        "valor": round(random.uniform(100, 1800), 2),
    }
    producer_vip.send(
        TOPIC_VIP,
        key=pedido["cliente_id"].encode("utf-8"),
        value=pedido,
    )

producer_vip.flush()
producer_vip.close()

consumer_vip = KafkaConsumer(
    TOPIC_VIP,
    bootstrap_servers=BOOTSTRAP_SERVERS,
    group_id=None,
    auto_offset_reset="earliest",
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    consumer_timeout_ms=5000,
)

mensagens_por_particao = Counter()
vips_por_particao = Counter()
for msg in consumer_vip:
    mensagens_por_particao[msg.partition] += 1
    if msg.value["valor"] > 1000:
        vips_por_particao[msg.partition] += 1
consumer_vip.close()

print("Mensagens por partição:", dict(sorted(mensagens_por_particao.items())))
print("Pedidos VIP por partição:", dict(sorted(vips_por_particao.items())))


Tópico 'pedidos-vip' criado.


/tmp/ipykernel_153/1160808152.py:58: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer_vip = KafkaConsumer(


Mensagens por partição: {0: 12, 1: 12, 2: 13, 3: 3}
Pedidos VIP por partição: {0: 12}


**Resposta da Atividade 1:** a principal desvantagem é que a partição 0 pode ficar sobrecarregada. Se os preços aumentarem e a maioria dos pedidos passar de R$ 1.000, quase todas as mensagens irão para a mesma partição. O limite por valor também pode ficar desatualizado com o tempo. No campo de prioridade, a classificação é mais direta e pode ser controlada pela aplicação, embora ele também possa causar sobrecarga se quase tudo for marcado como urgente.


### Atividade 2 — Hopping window (janelas sobrepostas)

**Objetivo:** a Seção 4.3 implementou uma tumbling window (janelas fixas, sem sobreposição). Agora você vai implementar uma **hopping window**: janelas de tamanho fixo, mas que avançam (fazem "hop") em intervalos menores que o próprio tamanho da janela — ou seja, a mesma mensagem pode cair em mais de uma janela.

**Tarefa:**
1. Escreva uma função `agregar_por_janela_hopping(topico, group_id, tamanho_janela_ms, intervalo_hop_ms)` baseada em `agregar_por_janela` (Seção 4.3), mas que:
   - Para cada mensagem, calcule **todas** as janelas de tamanho `tamanho_janela_ms` às quais aquele timestamp pertence, considerando que uma nova janela começa a cada `intervalo_hop_ms` (não a cada `tamanho_janela_ms`, como na tumbling);
   - Some o valor do pedido a **cada uma** dessas janelas (não só a uma).
2. Rode com `tamanho_janela_ms=4000` e `intervalo_hop_ms=1000` sobre o tópico `pedidos-prioridade`, e compare o resultado com a tumbling window da Seção 4.3.

**Dica:** para um timestamp `t`, o início da janela hopping mais recente que o contém é `(t // intervalo_hop_ms) * intervalo_hop_ms`; a partir daí, é preciso considerar também os inícios de janela anteriores que ainda "alcançam" `t` (isto é, `inicio + tamanho_janela_ms > t`).

**Pergunta para o relatório:** com `tamanho_janela_ms=4000` e `intervalo_hop_ms=1000`, quantas janelas, no máximo, uma única mensagem pode afetar? Generalize essa conta para `tamanho_janela_ms` e `intervalo_hop_ms` quaisquer.


In [11]:
# Atividade 2
def agregar_por_janela_hopping(topico, group_id, tamanho_janela_ms, intervalo_hop_ms):
    consumer = KafkaConsumer(
        topico,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=group_id,
        auto_offset_reset="earliest",
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms=5000,
    )

    totais = {}
    for msg in consumer:
        inicio = (msg.timestamp // intervalo_hop_ms) * intervalo_hop_ms

        # Volta pelas janelas anteriores enquanto elas ainda contêm a mensagem.
        while inicio + tamanho_janela_ms > msg.timestamp:
            totais[inicio] = totais.get(inicio, 0.0) + msg.value["valor"]
            inicio -= intervalo_hop_ms

    consumer.close()
    return dict(sorted(totais.items()))


totais_hopping = agregar_por_janela_hopping(
    TOPIC,
    None,
    tamanho_janela_ms=4000,
    intervalo_hop_ms=1000,
)

print("Tumbling window (2 segundos):")
for inicio, total in totais_janela.items():
    print(inicio, "->", f"R$ {total:.2f}")

print("\nHopping window (4 segundos, hop de 1 segundo):")
for inicio, total in totais_hopping.items():
    print(inicio, "->", f"R$ {total:.2f}")


/tmp/ipykernel_153/1179973855.py:3: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Tumbling window (2 segundos):
1790215656000 -> R$ 11450.47

Hopping window (4 segundos, hop de 1 segundo):
1790215653000 -> R$ 11450.47
1790215654000 -> R$ 11450.47
1790215655000 -> R$ 11450.47
1790215656000 -> R$ 11450.47


**Resposta da Atividade 2:** com janela de 4.000 ms e hop de 1.000 ms, uma mensagem pode entrar em no máximo **4 janelas**. No geral, o máximo é `ceil(tamanho_janela_ms / intervalo_hop_ms)`. A hopping window repete a mesma mensagem nas janelas sobrepostas, enquanto a tumbling window coloca cada mensagem em apenas uma janela.


### Atividade 3 — Join KStream-KStream com janela de tempo

**Objetivo:** a Seção 4.4 implementou um join entre um stream de eventos e uma KTable (sem necessidade de janela). Agora você vai implementar o tipo de join mais exigente: **KStream-KStream**, que sempre precisa de uma janela de tempo, porque estamos correlacionando dois eventos independentes.

**Cenário:** dois tópicos, `cliques-anuncio` (evento: usuário clicou num anúncio) e `compras` (evento: usuário completou uma compra). Você quer identificar quais compras aconteceram **dentro de uma janela de tempo** após um clique do mesmo usuário (uma atribuição de conversão simplificada).

**Tarefa:**
1. Crie os dois tópicos (`cliques-anuncio` e `compras`), cada um com o campo `usuario_id` e um `timestamp` (pode ser o próprio `msg.timestamp` do Kafka).
2. Publique alguns cliques e algumas compras nesses tópicos — inclua propositalmente: (a) um usuário que clicou e comprou logo em seguida (deve ser correlacionado), (b) um usuário que só clicou, sem comprar (não deve aparecer no resultado), e (c) um usuário que comprou sem ter clicado antes (não deve aparecer no resultado).
3. Escreva uma função `join_cliques_compras(janela_ms)` que:
   - Consome todos os cliques e todas as compras (dois `KafkaConsumer` separados, ou um de cada vez);
   - Para cada compra, procura se existe um clique do **mesmo `usuario_id`**, com `timestamp` do clique **anterior** ao da compra e dentro da janela (`timestamp_compra - timestamp_clique <= janela_ms`);
   - Retorna a lista de pares (clique, compra) que se qualificam como uma "conversão".

**Pergunta para o relatório:** o que aconteceria com o resultado se o mesmo usuário tivesse clicado **duas vezes** antes de comprar (ambos os cliques dentro da janela)? Sua implementação retornaria uma conversão ou duas? Qual das duas opções faz mais sentido de negócio, na sua opinião?


In [12]:
# Atividade 3
TOPIC_CLIQUES = "cliques-anuncio"
TOPIC_COMPRAS = "compras"

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP_SERVERS)
topicos_existentes = set(admin.list_topics())
novos_topicos = []
for nome in [TOPIC_CLIQUES, TOPIC_COMPRAS]:
    if nome not in topicos_existentes:
        novos_topicos.append(NewTopic(name=nome, num_partitions=1, replication_factor=1))
if novos_topicos:
    admin.create_topics(novos_topicos)
admin.close()
print("Tópicos de cliques e compras prontos.")

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    key_serializer=lambda k: k.encode("utf-8"),
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

# Os timestamps são definidos para deixar o teste previsível.
agora = int(time.time() * 1000)
cliques_teste = [
    ({"usuario_id": "usuario-1", "anuncio": "A"}, agora),
    ({"usuario_id": "usuario-2", "anuncio": "B"}, agora + 200),
    ({"usuario_id": "usuario-4", "anuncio": "C"}, agora + 300),
    ({"usuario_id": "usuario-4", "anuncio": "D"}, agora + 700),
]
compras_teste = [
    ({"usuario_id": "usuario-1", "valor": 120.0}, agora + 1000),
    ({"usuario_id": "usuario-3", "valor": 80.0}, agora + 1200),
    ({"usuario_id": "usuario-4", "valor": 250.0}, agora + 1500),
]

for clique, timestamp in cliques_teste:
    producer.send(
        TOPIC_CLIQUES,
        key=clique["usuario_id"],
        value=clique,
        timestamp_ms=timestamp,
    )
for compra, timestamp in compras_teste:
    producer.send(
        TOPIC_COMPRAS,
        key=compra["usuario_id"],
        value=compra,
        timestamp_ms=timestamp,
    )
producer.flush()
producer.close()


def ler_eventos(topico):
    consumer = KafkaConsumer(
        topico,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=None,
        auto_offset_reset="earliest",
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms=5000,
    )
    eventos = []
    for msg in consumer:
        eventos.append({**msg.value, "timestamp": msg.timestamp})
    consumer.close()
    return eventos


def join_cliques_compras(janela_ms):
    cliques = ler_eventos(TOPIC_CLIQUES)
    compras = ler_eventos(TOPIC_COMPRAS)
    conversoes = []

    for compra in compras:
        for clique in cliques:
            mesmo_usuario = clique["usuario_id"] == compra["usuario_id"]
            diferenca = compra["timestamp"] - clique["timestamp"]
            dentro_da_janela = 0 <= diferenca <= janela_ms

            if mesmo_usuario and dentro_da_janela:
                conversoes.append((clique, compra))

    return conversoes


conversoes = join_cliques_compras(janela_ms=2000)
for clique, compra in conversoes:
    print("Clique:", clique)
    print("Compra:", compra)
    print("---")
print("Total de conversões encontradas:", len(conversoes))


Tópicos de cliques e compras prontos.


/tmp/ipykernel_153/2184990239.py:55: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Clique: {'usuario_id': 'usuario-1', 'anuncio': 'A', 'timestamp': 1790215699511}
Compra: {'usuario_id': 'usuario-1', 'valor': 120.0, 'timestamp': 1790215700511}
---
Clique: {'usuario_id': 'usuario-4', 'anuncio': 'C', 'timestamp': 1790215699811}
Compra: {'usuario_id': 'usuario-4', 'valor': 250.0, 'timestamp': 1790215701011}
---
Clique: {'usuario_id': 'usuario-4', 'anuncio': 'D', 'timestamp': 1790215700211}
Compra: {'usuario_id': 'usuario-4', 'valor': 250.0, 'timestamp': 1790215701011}
---
Total de conversões encontradas: 3


**Resposta da Atividade 3:** esta implementação retorna **duas conversões** se os dois cliques do mesmo usuário estiverem dentro da janela antes da compra, pois testa todos os pares possíveis. Para este caso, considero melhor contar somente uma conversão por compra e atribuí-la ao clique mais recente. Isso evita contar a mesma venda duas vezes. Mesmo assim, mantive o código seguindo literalmente a ideia de retornar todos os pares que atendem à condição.


## 6. Extra: como isso fica com o Kafka Streams de verdade (e uma alternativa em Python)

Tudo o que implementamos manualmente nas Seções 4 e 5 (KTable, agregação, janelas, joins) tem equivalente direto na DSL do Kafka Streams — só que declarativo, sem escrever o loop de consumo à mão. Não é necessário instalar nada disso para esta aula — é só para comparação.

```java
// Equivalente Java (Kafka Streams DSL) ao que fizemos manualmente:

KStream<String, Pedido> pedidos = builder.stream("pedidos-prioridade");
KTable<String, Cliente> clientes = builder.table("clientes-cadastro");

// Agregação contínua (equivalente à Seção 4.2)
KTable<String, Double> totalPorCliente = pedidos
    .groupBy((chave, pedido) -> pedido.getClienteId())
    .aggregate(() -> 0.0, (clienteId, pedido, total) -> total + pedido.getValor());

// Tumbling window (equivalente à Seção 4.3)
KTable<Windowed<String>, Double> totalPorJanela = pedidos
    .groupBy((chave, pedido) -> pedido.getClienteId())
    .windowedBy(TimeWindows.ofSizeWithNoGrace(Duration.ofSeconds(2)))
    .aggregate(() -> 0.0, (clienteId, pedido, total) -> total + pedido.getValor());

// Join KStream-KTable (equivalente à Seção 4.4)
KStream<String, PedidoEnriquecido> pedidosEnriquecidos = pedidos.join(
    clientes,
    (pedido, cliente) -> new PedidoEnriquecido(pedido, cliente)
);
```

**O que a biblioteca resolve para você, automaticamente:**
- Gerencia os *state stores* (RocksDB) de forma durável e tolerante a falhas — nosso `dict` em memória some se o processo cair; o state store do Kafka Streams sobrevive, com *changelog topics* de backup;
- Cuida de *rebalanceamento* de partições entre instâncias da aplicação automaticamente (nosso código sempre rodou como um único processo, lendo o tópico inteiro);
- Lida com dados fora de ordem via *watermarks* e período de graça configurável, algo que nossa implementação manual de tumbling/hopping window ignora por completo.

**Uma alternativa que roda de verdade em Python:** a biblioteca [`faust`](https://faust.readthedocs.io/) foi explicitamente inspirada nos mesmos conceitos do Kafka Streams (chama suas unidades de processamento de "agents", tem suporte a tabelas e janelas), e pode ser instalada com `pip install faust-streaming`. Não faz parte do escopo desta aula, mas é a opção mais direta se você quiser, no futuro, processar streams Kafka em Python sem reimplementar os mecanismos à mão como fizemos aqui.
